# W05A · AE/VAE 표현 관찰실
2026-09-28 · 생성형 인공지능 · 주교재 3장

A는 복원·잠재 분포·생성 실험입니다. MNIST 약 11.6 MB를 처음 다운로드하고 캐시를 재사용합니다. 지난 W04A를 마치지 않았어도 새 사본에서 시작할 수 있습니다.
CPU 기본, API 키·개인 파일 업로드 불필요. 패키지 **0.1.7**, 고정 태그 **2026-fall-w05a-v2**.
[강의노트](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/course/notion/w05a/w05a-image-representations.md) · [기록지](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/course/notion/w05a/w05a-workbook.md) · [전체 API·소스 바로가기](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/W05A-API.md)

빈칸은 별도 연습 변수에 적습니다. None을 남겨도 기본 관찰을 막지 않습니다.
기본 코드의 결과를 먼저 예측하고 실행한 다음, 마지막 점검 셀까지 확인하세요.


## 0. 환경 준비
설치는 패키지와 수업 체크포인트·사진을 함께 준비합니다. 모델이나 데이터는 필요한 준비 셀에서 받습니다.
이미 import한 패키지를 교체하려면 런타임을 재시작하세요. 패키지 버전과 실제 Python 버전을 출력합니다.


In [ ]:
import sys, subprocess, os
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
PUBLIC_REF = '2026-fall-w05a-v2'
try:
    ready = version('luna-genai') == '0.1.7' and version('transformers') == '5.15.1' and version('gradio') == '6.26.0'
except PackageNotFoundError:
    ready = False
if not ready:
    if any(x in sys.modules for x in ('luna_genai','torch','transformers','gradio')):
        raise RuntimeError('런타임을 재시작하고 첫 설치 셀부터 실행하세요.')
    package = str(Path('src').resolve()) if Path('src/pyproject.toml').exists() else f'git+https://github.com/lunalab-ai/genAI.git@{PUBLIC_REF}#subdirectory=src'
    subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
print('Python',sys.version.split()[0],'luna-genai',version('luna-genai'),'transformers',version('transformers'))
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
torch.set_num_threads(2)


## 1. 데이터와 체크포인트를 준비하기
[`load_mnist`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/autoencoder.py#L92)는 CPU float32 `(N,1,28,28)`의 `MNISTSplit`을 반환합니다.
이번 호출은 train=20000, validation=2000, test=1000, seed=1337입니다. train/validation은 공식 train의 서로 다른 인덱스입니다.
[`load_checkpoints`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L190)는 가중치 SHA-256을 검사해 모델 사전과 학습 기록을 반환합니다. 학습이나 재다운로드를 수행하지 않습니다.
사진·가중치는 패키지 설치에 포함되고 MNIST만 별도로 자동 준비합니다. test_labels는 관찰용이며 학습에 쓰지 않습니다.


In [ ]:
from luna_genai.autoencoder import load_mnist
from luna_genai.representation import (DenseAE,DenseVAE,load_checkpoints,reconstruct_images,
    latent_means,reparameterize,vae_loss,fit_representation,reconstruction_figure,
    latent_figure,latent_grid,interpolation)
data=load_mnist(train_size=20000,validation_size=2000,test_size=1000,seed=1337)
models,provenance=load_checkpoints()
print('이번에 받은 MNIST 파일:',data.downloads or '없음: 검증된 캐시 재사용')
print('train/validation/test:',data.train.shape,data.validation.shape,data.test.shape)
print('인덱스 중복:',len(np.intersect1d(data.train_ids,data.validation_ids)))
display(pd.DataFrame([{ 'model':name,'dimension':m['latent_dim'],'beta':m['beta'],
                       'epochs':provenance['epochs'],'recorded_test_mse':m['test_mse']}
                      for name,m in provenance['models'].items()]))


### A1 · 입력과 target
목표: 배치 축과 자기 지도 target 구별. `(32,1,28,28)`의 각 숫자를 설명하고, 입력이 숫자 7이면 target이 무엇인지 적으세요. 힌트: 이미지 모양과 숫자 이름을 분리합니다. 자가 점검: target shape이 복원과 같아야 합니다.


## 2. MSE와 학습 한 단계
[`DenseAE`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L21)는 `784→256→128→d` 인코더와 반대 방향 디코더입니다. `latent_dim=2`, `seed=1337` 기본값이며 초기화만 합니다.
[`DenseAE.encode`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L42)의 출력은 `(B,d)`, [`DenseAE.decode`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L46)의 출력은 `(B,1,28,28)`입니다.
`torch.nn.functional.mse_loss`는 모든 원소 평균입니다. [공식 MSE 설명](https://docs.pytorch.org/docs/2.8/generated/torch.nn.MSELoss.html).
아래 4픽셀 예는 설명용이고 MNIST 측정값이 아닙니다.


In [ ]:
toy_x=torch.tensor([[0.,1.,.5,.5]])
toy_prediction=torch.tensor([[.2,.8,.4,.6]])
print('제곱 오차:',(toy_prediction-toy_x).square())
print('합:',(toy_prediction-toy_x).square().sum().item())
print('평균:',torch.nn.functional.mse_loss(toy_prediction,toy_x).item())


### A2 · 손실 빈칸과 디버깅
목표: target·분모 구별. 아래 연습 변수에 MSE를 계산해 넣고, 결과가 0.1인 코드에서 빠진 연산을 설명하세요. 힌트: 총 픽셀 수. 자가 점검: 0보다 크고 0.1보다 작습니다.


In [ ]:
my_mse=None  # 연습: 위 두 텐서로 평균 제곱 오차 계산
if my_mse is not None:
    print('내 계산:',float(my_mse))
else:
    print('연습 답을 적은 뒤 이 셀만 다시 실행할 수 있습니다.')


In [ ]:
fresh=DenseAE(2)
x_batch=data.train[:64]
optimizer=torch.optim.Adam(fresh.parameters(),lr=.001)
before=next(fresh.parameters()).detach().clone()
optimizer.zero_grad(set_to_none=True)
prediction=fresh(x_batch)
loss=torch.nn.functional.mse_loss(prediction,x_batch)
loss.backward()
after_backward=next(fresh.parameters()).detach().clone()
gradient_norm=next(fresh.parameters()).grad.norm().item()
optimizer.step()
parameter_change=(next(fresh.parameters()).detach()-before).abs().sum().item()
print('gradient norm:',gradient_norm,'backward 뒤 모수 동일:',torch.equal(before,after_backward))
print('step 뒤 모수 변화 합:',parameter_change)


### A3 · 학습과 평가 상태
목표: 미분 계산·모수 갱신 구별. backward 뒤와 step 뒤를 설명하고 eval만으로 gradient 기록이 꺼지는지 답하세요. 힌트: 위 출력의 두 시점. 자가 점검: 모수 변화 합을 확인합니다.


## 3. 같은 test로 복원 비교
[`reconstruct_images`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L143)는 배치별 추론으로 순서를 보존한 CPU 텐서를 반환합니다. VAE는 평균 μ를 디코딩합니다.
[`reconstruction_figure`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L210)는 원본과 모델별 복원을 같은 밝기 범위로 그립니다. 비교 지표는 모든 모델에서 같은 픽셀 MSE입니다.
평균 기준선은 train에서 만든 한 장을 모든 test에 반복합니다. 아래 표와 그림을 함께 읽으세요.


In [ ]:
baseline=float((data.test-data.train.mean(0,keepdim=True)).square().mean())
rows=[dict(model='train-mean baseline',test_mse=baseline)]
for name,model in models.items():
    rows.append(dict(model=name,test_mse=float((reconstruct_images(model,data.test)-data.test).square().mean())))
comparison=pd.DataFrame(rows)
display(comparison)
fig=reconstruction_figure({'AE2':models['ae2'],'AE16':models['ae16'],'VAE beta1':models['vae1']},data.test)
display(fig);plt.close(fig)


### A4 · 복원 해석
목표: 숫자와 그림의 근거 결합. AE2와 AE16에서 틀린 획의 위치를 하나씩 찾고, 낮은 MSE가 모든 숫자의 정확한 복원을 뜻하는지 설명하세요. 힌트: 첫 8장 세로 비교. 자가 점검: 관찰한 이미지 번호와 위치를 적습니다.


## 4. VAE의 평균·로그분산·샘플
[`DenseVAE.encode`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L77)는 `(mu,logvar)`를 반환합니다. [`reparameterize`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L97)는 `mu + exp(0.5*logvar)*epsilon`을 계산하며 epsilon 생략 시 표준정규 잡음을 뽑습니다.
아래 수계산은 고정 epsilon을 주므로 재현됩니다. $q(z|x)$는 입력별 대각 정규분포이고 prior는 $\mathcal{N}(0,I)$입니다.


In [ ]:
mu=torch.tensor([[1.,-1.]])
logvar=torch.log(torch.tensor([[4.,.25]]))
epsilon=torch.tensor([[.5,-2.]])
print('분산:',logvar.exp(),'표준편차:',(.5*logvar).exp())
print('고정 잡음의 z:',reparameterize(mu,logvar,epsilon))
with torch.inference_mode():
    actual_mu,actual_lv=models['vae1'].encode(data.test[:3])
print('실제 3장 평균:',actual_mu,'실제 표준편차:',(.5*actual_lv).exp())


### A5 · 로그분산 빈칸
목표: 분산·표준편차 구별. 연습 변수에 logvar를 표준편차로 바꾼 값을 넣으세요. `exp(logvar)`를 곱했을 때 무엇이 잘못되는지 설명하세요. 힌트: 분산의 제곱근. 자가 점검: 첫 좌표의 표준편차는 분산보다 작습니다.


In [ ]:
my_std=None  # 연습: logvar에서 표준편차 계산
if my_std is not None: print('내 z:',mu+my_std*epsilon)


## 5. 손실 두 항과 KL 가중치
[`vae_loss`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L115)는 `(total,sse,kl)`를 반환합니다. 이미지는 배치 평균, 픽셀과 잠재 좌표는 이미지 안에서 합입니다.
MNIST의 SSE는 픽셀 MSE의 784배입니다. 아래 4픽셀 예는 SSE 0.1, KL 2.125입니다.


In [ ]:
total,sse,kl=vae_loss(toy_prediction.reshape(1,1,2,2),toy_x.reshape(1,1,2,2),mu,logvar,beta=1.)
print('SSE / KL / total:',sse.item(),kl.item(),total.item())
display(pd.DataFrame([dict(model=n,beta=v['beta'],train_kl=v['history'][-1]['train_kl'],
                          validation_mse=v['history'][-1]['val_mse'])
                      for n,v in provenance['models'].items() if v['kind']=='VAE']))


### A6 · 손실 스케일과 β
목표: 공정한 비교. MNIST의 복원항을 합에서 평균으로 바꾸고 β를 그대로 두면 KL의 상대 영향은 어떻게 되는가요? β가 큰 모델의 복원이 반드시 더 선명한가요? 힌트: 784개 픽셀. 자가 점검: 같은 단위끼리 비교합니다.


## 6. 잠재 좌표, 보간, 새로운 샘플
[`latent_means`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L135)는 AE의 z 또는 VAE의 μ를 반환합니다. [`latent_figure`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L224)의 라벨 색은 관찰용입니다.
[`latent_grid`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L237)는 기본 [-3,3] 격자를 디코딩하며 정규분포 샘플이 아닙니다. [`interpolation`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L249)은 두 이미지의 z/μ 사이를 9단계로 연결합니다.


In [ ]:
fig=latent_figure({'AE2':models['ae2'],'VAE mean':models['vae1']},data.test,data.test_labels)
display(fig);plt.close(fig)
fig,axes=plt.subplots(1,3,figsize=(12,4))
for ax,name in zip(axes,['vae01','vae1','vae4']):
    ax.imshow(latent_grid(models[name]),cmap='gray',vmin=0,vmax=1);ax.set_title(name);ax.axis('off')
display(fig);plt.close(fig)
steps=interpolation(models['vae1'],data.test[:1],data.test[1:2])
fig,axes=plt.subplots(1,len(steps),figsize=(12,2))
for ax,pic in zip(axes,steps):ax.imshow(pic[0],cmap='gray',vmin=0,vmax=1);ax.axis('off')
display(fig);plt.close(fig)


### A7 · 생성과 보간 해석
목표: 샘플을 뽑는 규칙 구별. AE에 표준정규 z를 넣는 경우와 VAE에 넣는 경우가 왜 다른가요? 평균 산점도가 정확한 정규분포여야 하나요? 힌트: 목적함수의 KL 항. 자가 점검: q(z|x), prior, 평균 점을 구별합니다.


## 7. 선택: 직접 조금 더 학습하기
[`fit_representation`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/representation.py#L155)는 모델 모수를 바꾸고 epoch별 학습 항·validation MSE·시간을 반환합니다. 기본 epochs=5, batch_size=256, lr=0.001, seed=1337입니다.
아래 기본값은 False입니다. 선택하면 별도의 새 모델을 2 epoch만 학습합니다. 제공된 50 epoch 모델을 재현했다는 뜻은 아닙니다. test는 학습에 넣지 않습니다.


In [ ]:
RUN_SHORT_TRAINING=False
if RUN_SHORT_TRAINING:
    my_model=DenseVAE(2)
    my_history=fit_representation(my_model,data.train[:2000],data.validation[:256],epochs=2,beta=1.)
    display(pd.DataFrame(my_history))
else:
    print('선택 재학습은 건너뜀. 기본 관찰은 검증된 50 epoch 체크포인트를 사용합니다.')


## 8. 마지막: 복원·잠재 공간 웹 앱
[`build_embedding_app`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/embedding_app.py#L42)는 준비된 models와 test로 Gradio Blocks를 만듭니다. [`reconstruction_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/embedding_app.py#L12)는 원본·복원·절대 잔차·MSE를, [`latent_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a-v2/src/luna_genai/embedding_app.py#L28)는 좌표의 디코딩 결과를 반환합니다.
기본 앱을 확인한 뒤 A8 확장 활동을 해 보세요. [Gradio 이벤트 설명](https://www.gradio.app/guides/blocks-and-event-listeners).
Colab에서는 공유 URL을 출력합니다. 링크는 런타임이 살아 있을 때만 유지됩니다.


In [ ]:
from luna_genai.embedding_app import build_embedding_app,reconstruction_view,latent_view,APP_CSS
app=build_embedding_app(models,data.test)
print(reconstruction_view('ae16',0,models,data.test)[3])
print(latent_view('vae1',0.,0.,models)[1])


### A8 · 직접 조립: 무작위 잠재 샘플 버튼
목표: 새 UI·함수·이벤트 연결. 기본 앱 뒤에 새 탭을 추가해 seed 입력과 생성 버튼, 이미지 출력을 연결하세요. 기본 앱은 수정하지 않아도 동작합니다. 힌트: Generator, randn((1,2)), vae1.decode, Button.click. 자가 점검: 같은 seed는 같은 이미지, 다른 seed는 다른 좌표입니다.


In [ ]:
# 연습용 확장 코드는 이 셀에 작성하세요. 기본 app은 위에서 준비되어 있습니다.
app.launch(share=IN_COLAB,prevent_thread_lock=True,inline=IN_COLAB,quiet=True,css=APP_CSS)


In [ ]:
assert data.test.shape==(1000,1,28,28)
assert np.intersect1d(data.train_ids,data.validation_ids).size==0
assert comparison.loc[comparison.model=='ae2','test_mse'].iloc[0]<baseline
assert comparison.loc[comparison.model=='ae16','test_mse'].iloc[0]<.02
assert gradient_norm>0 and parameter_change>0 and torch.equal(before,after_backward)
torch.testing.assert_close(reparameterize(mu,logvar,epsilon),torch.tensor([[2.,-2.]]))
assert abs(kl.item()-2.125)<1e-5 and abs(sse.item()-.1)<1e-5
original,restored,residual,metrics=reconstruction_view('vae1',0,models,data.test)
assert original.shape==restored.shape==residual.shape==(28,28)
assert np.isfinite(metrics['pixel_mse']) and not np.array_equal(original,restored)
assert app.is_running
from gradio_client import Client
http_original,http_restored,http_residual,http_metrics=Client(app.local_url,verbose=False).predict('ae16',0,api_name='/reconstruct')
assert all(Path(p).is_file() for p in (http_original,http_restored,http_residual))
assert np.isfinite(http_metrics['pixel_mse'])
if not IN_COLAB: app.close()
print('W05A A runtime checks passed: 실제 데이터·모델·수치·callback·서버 시작')


이 점검은 자동 계산과 서버 시작을 확인합니다. 화면의 버튼을 눌러 결과가 바뀌는지도 직접 확인하세요.
다음 [Colab B · CLIP 검색과 누적 앱](https://colab.research.google.com/github/lunalab-ai/genAI/blob/2026-fall-w05a-v2/notebooks/student/w05a_clip_search.ipynb)은 독립 실행 가능합니다.
